# Hypothesis 2 — Main Notebook

Hypothesis: The underlying LLM of an MAAI statistically significantly affects its fairness judgments; specifically, American and Chinese LLMs differ in their judgments.

This notebook prepares two groups of aligned experiment configurations (American, Chinese),
runs them selectively in parallel, and compares the outcome distributions across groups (5 categories).

- 34 configs per group (default), total 68.
- Per-config: a shared temperature drawn from U(0, 1.5), shared random seed, and agent models selected per group.
- Language is English for all runs.
- Utility agent is `google/gemini-2.5-flash`.
- Voting detection mode is set to "complex" for all runs.
- Exact test: Fisher–Freeman–Halton via R (if available). No Chi-square fallback.
- Effect size: Cramér's V (bias-corrected, as in Hypothesis 1).


## Model Overview (Used in this experiment)

The following OpenRouter models are used, grouped by country of origin.

| Model | Creator | Country | Intelligence Index | OpenRouter ID | Included |
|---|---|---|---|---|---|
| Gemini 2.5 Pro | Google | USA | 65 | google/gemini-2.5-pro | Yes |
| gpt-oss-120B (high) | OpenAI | USA | 61 | openai/gpt-oss-120b | Yes |
| Gemini 2.5 Flash (Reasoning) | Google | USA | 58 | google/gemini-2.5-flash | Yes |
| Grok 3 mini Reasoning (high) | xAI | USA | 58 | x-ai/grok-3-mini | Yes |
| GPT-4.1. | OpenAI | USA | 47 | openai/gpt-4.1 | Yes |
| gpt-oss-20B (high) | OpenAI | USA | 49 | openai/gpt-oss-20b | Yes |
| Qwen3 235B 2507 (Reasoning) | Alibaba | China | 64 | qwen/qwen3-235b-a22b-thinking-2507 | Yes |
| DeepSeek V3.1 (Reasoning) | DeepSeek | China | 60 | deepseek/deepseek-chat-v3.1 | Yes |
| DeepSeek R1 0528 | DeepSeek | China | 59 | deepseek/deepseek-r1-0528 | Yes |
| GLM-4.5 | Zhipu AI | China | 56 | z-ai/glm-4.5 | Yes |
| Qwen3 30B 2507 (Reasoning) | Alibaba | China | 54 | qwen/qwen3-30b-a3b-instruct-2507 | Yes |
| MiniMax M1 80k | MiniMax | China | 53 | minimax/minimax-m1 | Yes |


In [9]:
# Imports
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'hypothesis_testing').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
import numpy as np
from collections import Counter
from hypothesis_testing.utils_hypothesis_testing.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


In [10]:
# Base paths and groups
BASE_DIR = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_2'
CONFIGS_BASE = BASE_DIR / 'configs'
LOGS_BASE = BASE_DIR / 'terminal_outputs'
RESULTS_BASE = BASE_DIR / 'results'

GROUPS = {
    'american': 'American LLMs',
    'chinese': 'Chinese LLMs',
}

# Ensure subfolders exist
for key in GROUPS.keys():
    (CONFIGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (LOGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (RESULTS_BASE / key).mkdir(parents=True, exist_ok=True)
CONFIGS_BASE, LOGS_BASE, RESULTS_BASE, GROUPS


(PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/configs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/terminal_outputs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/results'),
 {'american': 'American LLMs', 'chinese': 'Chinese LLMs'})

## 1) Config Generation

Generates aligned YAML configurations for each group.
- Language is English for all runs.
- Utility agent: `google/gemini-2.5-flash`.
- Voting detection mode: `complex`.


In [12]:
# Income class probabilities (must sum to 1.0)
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# American and Chinese model pools (OpenRouter IDs)
AMERICAN_MODELS = [
    'google/gemini-2.5-pro',
    'openai/gpt-oss-120b',
    'google/gemini-2.5-flash',
    'x-ai/grok-3-mini',
    'openai/gpt-4.1',
    'openai/gpt-oss-20b',
]
CHINESE_MODELS = [
    'qwen/qwen3-235b-a22b-thinking-2507',
    'deepseek/deepseek-chat-v3.1',
    'deepseek/deepseek-r1-0528',
    'z-ai/glm-4.5',
    'qwen/qwen3-30b-a3b-instruct-2507',
    'minimax/minimax-m1',
]

def make_agents_with_models(temp: float, models: list[str]) -> list[dict]:
    agents = []
    for i in range(0, 4):  # 4 participant agents
        agents.append({
            'name': f'Agent_{i}',
            'personality': 'You are a college student',
            'model': models[i],
            'temperature': float(temp),
            'memory_character_limit': 25000,
            'reasoning_enabled': True,
        })
    return agents

def build_config(temp: float, seed_val: int, models: list[str]) -> dict:
    return {
        'language': 'English',
        'seed': int(seed_val),
        'agents': make_agents_with_models(temp, models),
        'utility_agent_model': 'google/gemini-2.5-flash',
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 10,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': { 'enabled': True },
        'voting_detection_mode': 'complex',
    }

# Optional: set a global seed for reproducible generation (adjust or comment out)
GLOBAL_SEED = 20000
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

def _pick_american_models() -> list[str]:
    # Allow repeats; sample each agent independently
    return [random.choice(AMERICAN_MODELS) for _ in range(4)]

def _pick_chinese_models() -> list[str]:
    return [random.choice(CHINESE_MODELS) for _ in range(4)]

def generate_aligned_configs(n: int = 34) -> dict[str, list[Path]]:
    paths: dict[str, list[Path]] = {k: [] for k in GROUPS.keys()}
    for idx in range(1, n + 1):
            temp = random.uniform(0.0, 1.5)
            seed_val = random.randint(0, 2**31 - 1)
            # Build per-group models
            models_american = _pick_american_models()
            models_chinese = _pick_chinese_models()

            # Write configs
            for group_key, models in [('american', models_american), ('chinese', models_chinese)]:
                cfg = build_config(temp=temp, seed_val=seed_val, models=models)
                out_dir = (CONFIGS_BASE / group_key)
                out_dir.mkdir(parents=True, exist_ok=True)
                fname = out_dir / f'hypothesis_2_{group_key}_condition_{idx}_config.yaml'
                with open(fname, 'w') as f:
                    yaml.safe_dump(cfg, f, sort_keys=False)
                paths[group_key].append(fname)
    return paths

# Example (commented):
# files_by_group = generate_aligned_configs(n=34)
# {k: len(v) for k, v in files_by_group.items()}


## 2) Run Configs (Parallel per Group)

Select subsets and run with per-group logs/results directories.


In [ ]:
def run_group(group_key: str, include_indices=None, include_names=None, concurrency: int = 4, timeout_sec: int | None = None):
    cfg_dir = CONFIGS_BASE / group_key
    logs_dir = LOGS_BASE / group_key
    results_dir = RESULTS_BASE / group_key
    configs = list_config_files(cfg_dir)
    selected = select_configs(configs, include_indices=include_indices, include_names=include_names)
    print(f'[{group_key}] Found {len(configs)} configs; selected {len(selected)}')
    run_results = run_configs_in_parallel(
        selected,
        concurrency=concurrency,
        logs_dir=logs_dir,
        results_dir=results_dir,
        timeout_sec=timeout_sec,
    )
    ok = sum(1 for r in run_results if r.get('ok'))
    print(f'[{group_key}] Completed: {ok}/{len(run_results)} OK')
    return run_results

# Example usage (uncomment to try small subsets):
# rr_us = run_group('american', include_indices=[1,2,3], concurrency=3)
# rr_cn = run_group('chinese', include_indices=[1,2,3], concurrency=3)


## 3) Analysis — Compare Outcomes Across Groups

Build a 5×2 contingency table (rows=principle/disagreement categories; columns=American/Chinese)
and run Fisher–Freeman–Halton exact test via R when available.
Compute Cramér's V with bias correction (as in Hypothesis 1). No Chi-square fallback is included.


In [ ]:
CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

def count_by_group() -> dict[str, Counter]:
    out: dict[str, Counter] = {}
    for k in GROUPS.keys():
        counts = Counter()
        result_files = sorted((RESULTS_BASE / k).glob('*_results.json'))
        for rp in result_files:
            counts[categorize_result(rp)] += 1
        for cat in CATEGORIES:
            counts.setdefault(cat, 0)
        out[k] = counts
    return out

group_counts = count_by_group()
for k, counts in group_counts.items():
    print(f'{k.capitalize()} counts:', dict(counts))

# Build contingency table: rows=categories, cols=[American, Chinese]
col_order = ['american', 'chinese']
contingency = np.vstack([[group_counts[col][cat] for col in col_order] for cat in CATEGORIES])
contingency, CATEGORIES, col_order


In [ ]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    """Run Fisher–Freeman–Halton test via R's fisher.test if available.
    Returns p-value or None if Rscript not found or fails.
    """
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow, ncol = contingency.shape
    r_code = f"""m <- matrix(c({r_matrix}), nrow={nrow}, ncol={ncol}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) { cat(f$p.value) } else { cat('NA') }
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except Exception:
        return None

p_ffh = fisher_freeman_halton_pvalue_r(contingency)
if p_ffh is None:
    print('R not available; skipping Fisher–Freeman–Halton exact test')
else:
    print(f'Fisher–Freeman–Halton exact test p-value: {p_ffh:.6f}')


In [ ]:
def cramers_v(contingency: np.ndarray) -> float:
    from scipy.stats import chi2_contingency
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, c - 1))))

def bias_corrected_cramers_v(contingency: np.ndarray) -> float:
    from scipy.stats import chi2_contingency
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    phi2 = chi2 / n
    r1 = r - 1
    c1 = c - 1
    phi2_corr = max(0.0, phi2 - (r1 * c1) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    denom = min(r_corr - 1, c_corr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))

cv = cramers_v(contingency)
cv_corr = bias_corrected_cramers_v(contingency)
print(f"Cramér's V (uncorrected): {cv:.4f}")
print(f"Cramér's V (bias-corrected): {cv_corr:.4f}")
